# 🦙 TechQA — Fine-tuning Llama 3.2-3B Instruct with Unsloth (QLoRA)

- **Môn học:** Statistical Learning (Học máy thống kê) — HCMUS  
- **Bài toán:** Question Answering trên tập dữ liệu IBM TechQA  
- **Mô hình nền tảng:** `unsloth/Llama-3.2-3B-Instruct` (4-bit QLoRA)  
- **Mục tiêu:** Fine-tune mô hình Llama 3.2-3B trên các cặp câu hỏi - câu trả lời kỹ thuật của TechQA, sau đó export bản 16-bit merge để phục vụ RAG QA Pipeline.

## 1. Cài đặt Thư viện & Cấu hình Môi trường
Cài đặt thư viện `unsloth` và các dependencies cần thiết trên Google Colab GPU (Tesla T4 / A100).

In [1]:
%%capture
!pip install --no-cache-dir --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir --upgrade git+https://github.com/unslothai/unsloth-zoo.git

## 2. Tải Base Model & Cấu hình LoRA Adapter (PEFT)
- Tải pre-quantized 4-bit model `unsloth/Llama-3.2-3B-Instruct` để tiết kiệm VRAM và tăng tốc độ tải.
- Áp dụng LoRA với rank $r=16$, $\alpha=16$, nhắm vào toàn bộ các linear projection layers (`q, k, v, o, gate, up, down`).

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Độ dài context tối đa hỗ trợ RoPE Scaling
dtype = None           # None để tự động detect (Float16 cho T4/V100, Bfloat16 cho Ampere+)
load_in_4bit = True    # 4-bit quantization giảm VRAM sử dụng

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,             # LoRA rank (8, 16, 32, 64)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,   # 0 là tối ưu nhất cho Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Tiết kiệm 30% VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 3. Tiền xử lý Dữ liệu TechQA (Dataset Preparation & Formatting)
- Đọc tập dữ liệu `training_Q_A.json` và `dev_Q_A.json`.
- Lọc chỉ lấy các câu hỏi có thể trả lời được (`ANSWERABLE == "Y"`).
- Định dạng hội thoại đa lượt theo chuẩn Llama 3.1 (`system`, `user`, `assistant`).

In [4]:
from unsloth.chat_templates import get_chat_template
import json
from datasets import Dataset

# Llama 3.2 kế thừa chat template của Llama 3.1
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# --- 1. Đọc dữ liệu TechQA từ file JSON ---
with open("training_Q_A.json", "r", encoding="utf-8") as f:
    train_raw = json.load(f)

with open("dev_Q_A.json", "r", encoding="utf-8") as f:
    dev_raw = json.load(f)

# --- 2. Chuyển TechQA JSON sang format Chat Conversations ---
def techqa_to_conversations(samples):
    conversations_list = []
    for item in samples:
        if item.get("ANSWERABLE") != "Y":
            continue

        question = item["QUESTION_TITLE"].strip()
        if item.get("QUESTION_TEXT", "").strip():
            question += "\n\n" + item["QUESTION_TEXT"].strip()

        answer = item["ANSWER"].strip()

        conversations_list.append({
            "conversations": [
                {
                    "role": "system",
                    "content": (
                        "You are a technical support assistant specialized in IBM products. "
                        "Answer the user's technical question accurately and concisely "
                        "based on your knowledge of IBM technotes and documentation."
                    ),
                },
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer},
            ]
        })
    return conversations_list

train_convos = techqa_to_conversations(train_raw)
dev_convos = techqa_to_conversations(dev_raw)

print(f"📊 Training samples (answerable): {len(train_convos)}")
print(f"📊 Dev/Validation samples (answerable): {len(dev_convos)}")

# --- 3. Tạo HuggingFace Dataset & Áp dụng Chat Template ---
dataset = Dataset.from_list(train_convos)
eval_dataset = Dataset.from_list(dev_convos)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)

📊 Training samples (answerable): 450
📊 Dev/Validation samples (answerable): 160


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

In [5]:
# Kiểm tra mẫu dữ liệu đầu tiên sau khi format
print("=== MẪU CONVERSATION (ITEM 0) ===")
print(dataset[0]["conversations"])
print("\n=== MẪU FORMATTED TEXT (ITEM 0) ===")
print(dataset[0]["text"][:1000])

=== MẪU CONVERSATION (ITEM 0) ===
[{'role': 'system', 'content': "You are a technical support assistant specialized in IBM products. Answer the user's technical question accurately and concisely based on your knowledge of IBM technotes and documentation."}, {'role': 'user', 'content': 'User environment variables no longer getting picked up after upgrade to 4.1.1.1 or 4.1.1.2?\n\nHave you found that after upgrade to Streams 4.1.1.1 or 4.1.1.2, that environment variables set in your .bashrc are no longer being set? For example ODBCINI is not set for the database toolkit and you get\n\n     An SQL operation failed. The SQL state is 08003, the SQL code\n     is 0 and the SQL message is [unixODBC][Driver\n     Manager]Connnection does not exist.'}, {'role': 'assistant', 'content': 'To work around the issue, set environment variables that are needed by the application directly in the instance with:  * \n   \n * streamtool setproperty\n * -d <domain> -i <instance>\n   --application-ev <VARIAB

## 4. Huấn luyện Mô hình với SFTTrainer (Supervised Fine-Tuning)
- Thiết lập SFTTrainer huấn luyện trong 3 Epochs với learning rate $2 \times 10^{-4}$ và Cosine Scheduler.
- Sử dụng kỹ thuật `train_on_responses_only` để mask prompt (chỉ tính loss trên câu trả lời của Assistant, không tính loss trên câu hỏi của User).

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Masking: chỉ tính loss trên response của assistant
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/450 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

In [7]:
# Hiển thị thống kê GPU trước khi huấn luyện
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"🖥️ GPU: {gpu_stats.name} | Dung lượng tối đa: {max_memory} GB")
print(f"💾 VRAM đang dùng: {start_gpu_memory} GB")

# Bắt đầu quá trình huấn luyện
print("\n🚀 Bắt đầu quá trình Fine-tuning...")
trainer_stats = trainer.train()

# Thống kê kết quả sau khi train
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n✅ Hoàn thành trong: {round(trainer_stats.metrics['train_runtime']/60, 2)} phút")
print(f"💾 Peak VRAM sử dụng: {used_memory} GB ({round(used_memory/max_memory*100, 2)}%)")

🖥️ GPU: Tesla T4 | Dung lượng tối đa: 14.563 GB
💾 VRAM đang dùng: 2.359 GB

🚀 Bắt đầu quá trình Fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 3 | Total steps = 171
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,2.257622,2.361526
2,1.770432,2.306351
3,1.433478,2.341847


Filter:   0%|          | 0/160 [00:00<?, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-57/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-114/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-171/tokenizer_config.json.



✅ Hoàn thành trong: 13.97 phút
💾 Peak VRAM sử dụng: 4.229 GB (29.04%)


## 5. Thử nghiệm Suy luận (Inference Test)
Kiểm tra chất lượng sinh câu trả lời kỹ thuật của mô hình vừa fine-tune với câu hỏi mẫu thực tế.

In [8]:
FastLanguageModel.for_inference(model)  # Kích hoạt chế độ suy luận 2x faster của Unsloth

messages = [
    {
        "role": "system",
        "content": (
            "You are a technical support assistant specialized in IBM products. "
            "Answer the user's technical question accurately and concisely "
            "based on your knowledge of IBM technotes and documentation."
        ),
    },
    {
        "role": "user",
        "content": "How to resolve an out-of-memory error when running WebSphere Application Server?",
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.7,
    min_p = 0.1,
)

print("🤖 Response from Fine-tuned Model:\n")
print(tokenizer.batch_decode(outputs)[0])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Response from Fine-tuned Model:

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a technical support assistant specialized in IBM products. Answer the user's technical question accurately and concisely based on your knowledge of IBM technotes and documentation.<|eot_id|><|start_header_id|>user<|end_header_id|>

How to resolve an out-of-memory error when running WebSphere Application Server?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The following steps can be taken to increase the amount of memory allocated to the JVM and resolve the Out of Memory (OOM) error:

 1. Increase the heap size: 
     * Add the following JVM argument to the start-up command line: 
       -Xms1024m -Xmx1024m 
       * where 1024 is the size in megabytes. 
     * If the server is configured to start multiple JVMs, the above command line argument should be added to the start-up command line of all the JVMs. 

 2. 

## 6. Xuất Mô hình Hoàn chỉnh (16-bit Merge) & Đóng gói Tải về / Đẩy lên Huggingface
- Merge trọng số LoRA vào Base Model và lưu dưới định dạng 16-bit safetensors chuẩn Hugging Face vào thư mục `Llama_TechQA`.
- Nén thư mục thành file `Llama_TechQA.zip` để tải về máy local và sử dụng trong thư mục `models/Llama_TechQA/` của dự án.

In [9]:
import shutil

# 1. Merge và lưu trọng số 16-bit
print("💾 Đang merge LoRA vào Base Model và lưu dưới dạng 16-bit safetensors...")
model.save_pretrained_merged("Llama_TechQA", tokenizer, save_method = "merged_16bit")

# 2. Đóng gói thư mục thành file zip để tải về
print("📦 Đang nén thư mục Llama_TechQA thành file zip...")
shutil.make_archive("Llama_TechQA", "zip", ".", "Llama_TechQA")

print("\n🎉 HOÀN TẤT 100%!")
print("📥 Bạn hãy tải file Llama_TechQA.zip từ sidebar Files của Colab về máy tính.")
print("📂 Giải nén file zip vào thư mục: qa_nlp_project/models/Llama_TechQA/")

💾 Đang merge LoRA vào Base Model và lưu dưới dạng 16-bit safetensors...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in Llama_TechQA/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [03:43<00:00, 111.93s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)




Unsloth: Merging weights into 16bit:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [02:18<02:18, 138.81s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:06<00:00, 93.02s/it]


Unsloth: Merge process complete. Saved to `/content/Llama_TechQA`
📦 Đang nén thư mục Llama_TechQA thành file zip...

🎉 HOÀN TẤT 100%!
📥 Bạn hãy tải file Llama_TechQA.zip từ sidebar Files của Colab về máy tính.
📂 Giải nén file zip vào thư mục: qa_nlp_project/models/Llama_TechQA/


In [10]:
# Đẩy thẳng toàn bộ model 16-bit lên Hugging Face Hub
model.push_to_hub_merged(
    "AQUABOT/Llama-3.2-3B-TechQA",
    tokenizer,
    save_method = "merged_16bit",
    token = "hf_writetoken",
    private = False,
)


Unsloth: Restored added_tokens_decoder metadata in AQUABOT/Llama-3.2-3B-TechQA/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  608kB / 1.46GB            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:46<00:00, 113.03s/it]


Unsloth: Merge process complete. Saved to `/content/AQUABOT/Llama-3.2-3B-TechQA`


In [11]:
from google.colab import drive
import shutil

# 1. Kết nối với Google Drive của bạn
drive.mount('/content/drive')

# 2. Copy thẳng file zip 6GB sang Google Drive
print("Đang copy sang Google Drive...")
shutil.copy("Llama_TechQA.zip", "/content/drive/MyDrive/Llama_TechQA.zip")
print("🎉 ĐÃ XONG! Mở Google Drive lên là thấy ngay file Llama_TechQA.zip!")

Mounted at /content/drive
Đang copy sang Google Drive...
🎉 ĐÃ XONG! Mở Google Drive của bạn lên là thấy ngay file Llama_TechQA.zip!
